In [1]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, Model
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [2]:
# ==========================================
# 1. LOAD THE MATRICES (IGNORING BMI)
# ==========================================
print("Loading Deep Learning Dataset (Pure Signal Only)...")
X_signals = np.load("X_signals.npy")  # Shape: (2150, 100, 2)
y_glucose = np.load("y_glucose.npy")  # Shape: (2150,)

# Split the data (80% for training, 20% for testing)
X_train, X_test, y_train, y_test = train_test_split(
    X_signals, y_glucose, test_size=0.2, random_state=42
)

Loading Deep Learning Dataset (Pure Signal Only)...


In [3]:
# ==========================================
# 2. BUILD THE PURE 1D-CNN ARCHITECTURE
# ==========================================
input_signal = layers.Input(shape=(100, 2), name="signal_input")

# Feature Extraction Layers
x = layers.Conv1D(filters=32, kernel_size=5, activation='relu')(input_signal)
x = layers.MaxPooling1D(pool_size=2)(x) 

x = layers.Conv1D(filters=64, kernel_size=3, activation='relu')(x)
x = layers.MaxPooling1D(pool_size=2)(x)

# Flatten into a 1D array
x = layers.Flatten()(x)

# Decision Layers
x = layers.Dense(64, activation='relu')(x)
x = layers.Dropout(0.2)(x) # Prevents memorization
x = layers.Dense(32, activation='relu')(x)

# Final Glucose Output
output = layers.Dense(1, activation='linear', name="glucose_output")(x)

# Compile Model
model = Model(inputs=input_signal, outputs=output)
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001), 
              loss='mse', 
              metrics=['mae'])

model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ signal_input (InputLayer)       │ (None, 100, 2)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d (Conv1D)                 │ (None, 96, 32)         │           352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d (MaxPooling1D)    │ (None, 48, 32)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_1 (Conv1D)               │ (None, 46, 64)         │         6,208 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_1 (MaxPooling1D)  │ (None, 23, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 1472)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 64)             │        94,272 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ glucose_output (Dense)          │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 102,945 (402.13 KB)

 Trainable params: 102,945 (402.13 KB)

 Non-trainable params: 0 (0.00 B)

In [4]:
# ==========================================
# 3. TRAIN THE MODEL
# ==========================================
print("\nTraining the Model on purely visual heartbeat features...")

# Stop early if it stops learning
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True)

history = model.fit(
    x=X_train, 
    y=y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32,
    callbacks=[early_stop],
    verbose=1
)


Training the Model on purely visual heartbeat features...
Epoch 1/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 7s 57ms/step - loss: 6261.8760 - mae: 64.5410 - val_loss: 1511.2574 - val_mae: 29.0246
Epoch 2/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1641.5242 - mae: 31.9662 - val_loss: 1364.2087 - val_mae: 28.9936
Epoch 3/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1571.1735 - mae: 31.1895 - val_loss: 1355.2053 - val_mae: 27.8057
Epoch 4/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1498.7484 - mae: 30.5223 - val_loss: 1401.1176 - val_mae: 27.3253
Epoch 5/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1453.7697 - mae: 29.4996 - val_loss: 1226.5801 - val_mae: 26.8051
Epoch 6/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1412.1149 - mae: 29.8192 - val_loss: 1246.6122 - val_mae: 26.1383
Epoch 7/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 1357.1631 - mae: 28.9869 - val_loss: 1135.7192 - val_mae: 26.4111
Epoch 8/100
54/54 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 1334.3

In [5]:
# ==========================================
# 4. EVALUATE THE RESULTS
# ==========================================
print("\n==================================================")
print("🏆 PURE SIGNAL DEEP LEARNING RESULTS")
print("==================================================")

predictions = model.predict(X_test).flatten()

mae = mean_absolute_error(y_test, predictions)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
r2 = r2_score(y_test, predictions)

print(f"➡️ Mean Absolute Error (MAE): {mae:.2f} mg/dL")
print(f"➡️ Root Mean Squared Error (RMSE): {rmse:.2f} mg/dL")
print(f"➡️ R-Squared (R2):            {r2:.2f}")


🏆 PURE SIGNAL DEEP LEARNING RESULTS
14/14 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step
➡️ Mean Absolute Error (MAE): 24.62 mg/dL
➡️ Root Mean Squared Error (RMSE): 31.72 mg/dL
➡️ R-Squared (R2):            0.18


In [8]:
# Save the pure model
model.save("glucose_1d_cnn_pure.h5")
print("\nModel saved successfully as 'glucose_1d_cnn_pure.h5'")


Model saved successfully as 'glucose_1d_cnn_pure.h5'
